In [1]:
import os

os.environ["OPENAI_API_KEY"] = ""

In [6]:
from datetime import date
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext

import nest_asyncio
nest_asyncio.apply()

class Vow(BaseModel):
    vow: str
    score: int | None = None

class SystemPromptSuggestion(BaseModel):
    original_prompt: str
    suggested_prompt: str
    explanation: str

vow_writer = Agent(
    'openai:gpt-4o',
    deps_type=str,
    result_type=Vow,
    system_prompt="Write a heartfelt and poetic wedding vow that expresses deep love and commitment. The vow should be personal, emotional and memorable."
)

vow_judge = Agent(
    'openai:gpt-4o', 
    deps_type=Vow,
    result_type=int,
    system_prompt="You are an expert judge of wedding vows. Rate the given wedding vow on a scale of 1 to 5, where 1 is poor and 5 is exceptional. Consider factors like emotional depth, poetic quality, originality, and sincerity."
)

opro = Agent(
    'openai:gpt-4o',
    deps_type=tuple[str, Vow],
    result_type=SystemPromptSuggestion,
    system_prompt="You are an expert at optimizing system prompts. Given a system prompt and the vow it generated (along with its score), suggest an improved version of the system prompt that would likely generate better wedding vows. Explain your reasoning."
)

# Example usage:
vow = vow_writer.run_sync("Write a wedding vow for someone who loves nature", deps="nature-loving")
print(f"Generated vow: {vow.data.vow}")

score = vow_judge.run_sync("Rate this vow", deps=vow.data)
vow.data.score = score.data
print(f"Vow score: {score.data}/5")

suggestion = opro.run_sync(
    "Suggest improvements to the system prompt",
    deps=(vow_writer.system_prompt, vow.data)
)
print(f"\nSuggested prompt improvement: {suggestion.data.suggested_prompt}")
print(f"Explanation: {suggestion.data.explanation}")

Generated vow: My Beloved,

Today, as we stand amidst the beauty of nature, surrounded by vibrant blooms and towering trees, I am reminded of the everlasting cycles of life and love.

Just like the sun rises each day to kiss the earth, I promise to wake each morning with gratitude for the gift of your love. Like the trees that reach for the sky to find light, I vow to always strive to uplift you and support your dreams.

In the way rivers carve their paths with gentle persistence, I promise to be patient and understanding, to adapt and grow with you, as we journey together down the river of life.

I am humbled by the depth of the ocean, and it is with that same depth that I commit my heart to you, wholly and unconditionally, casting away any fears like leaves in the wind.

Just as seasons change, I promise to love you through every winter and spring, every storm and sunshine, knowing that each moment only strengthens the roots of our bond.

With the song of the birds as our witness, I 